<a href="https://colab.research.google.com/github/manoj-naga-varma/Real-Time-Exercise-Tracker-and-Counter-in-Python/blob/main/PCL_Project_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pushups


In [ ]:
!pip install mediapipe opencv-python numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 11.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing in

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# 0
# Importing necessary libraries

# In[2]:


import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose


# In[4]:

#Calculating the angle between the necessary landmarks

def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)

    if angle >180.0:
        angle = 360-angle

    return angle


# # Pushups Counter

# In[22]:


#Pushups counter
cap = cv2.VideoCapture(0)

# Curl counter variables
counter = 0
stage = None

## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make detection
        results = pose.process(image)

        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark

            # Get coordinates
            shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
            l_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            l_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            l_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

            # Calculate angle
            angle = calculate_angle(shoulder, elbow, wrist)
            l_angle = calculate_angle(l_shoulder, l_elbow, l_wrist)
            avg_angle = (angle+l_angle)/2
            # Visualize angle
            cv2.putText(image, str(avg_angle),
                           tuple(np.multiply(elbow, [640, 480]).astype(int)),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA
                                )

            # Curl counter logic
            if avg_angle < 90:
                stage = "down"
            if avg_angle >= 160 and stage =='down':
                stage="up"
                counter +=1
                print(counter)

        except:
            pass

        # Render curl counter
        # Setup status box
        cv2.rectangle(image, (0,0), (325,73), (245,117,16),-1)

        # Rep data
        cv2.putText(image, 'PUSHUPS', (15,12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,220,0), 2,cv2.LINE_8)
        cv2.putText(image, str(counter),
                    (10,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2)

        # Stage data
        cv2.putText(image, 'STAGE', (128,12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,220,0), 2)
        cv2.putText(image, stage,
                    (125,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2)


        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(55,228,60), circle_radius=2),
                                mp_drawing.DrawingSpec(color=(255,153,204),circle_radius=2)
                                 )

        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
!pip install mediapipe==0.10.11 opencv-python==4.10.0.84 numpy==1.26.4 tensorflow==2.16.1

Reps


In [ ]:
#!/usr/bin/env python
# coding: utf-8

# 0
# Importing necessary libraries

# In[2]:


import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose


# In[4]:

#Calculating the angle between the necessary landmarks

def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)

    if angle >180.0:
        angle = 360-angle

    return angle


# 4
# Curl  Counter

# In[5]:


cap = cv2.VideoCapture(0)

# Curl counter variables
counter = 0
stage = None

## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make detection
        results = pose.process(image)

        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark

            # Get coordinates
            shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

            # Calculate angle
            angle = calculate_angle(shoulder, elbow, wrist)

            # Visualize angle
            cv2.putText(image, str(angle),
                           tuple(np.multiply(elbow, [640, 480]).astype(int)),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA
                                )

            # Curl counter logic
            if angle > 160:
                stage = "down"
            if angle < 45 and stage =='down':
                stage="up"
                counter +=1
                print(counter)

        except:
            pass

        # Render curl counter
        # Setup status box
        cv2.rectangle(image, (0,0), (325,73), (245,117,16),-1)

        # Rep data
        cv2.putText(image, 'REPS', (15,12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,220,0), 2,cv2.LINE_8)
        cv2.putText(image, str(counter),
                    (10,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2)

        # Stage data
        cv2.putText(image, 'STAGE', (128,12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,220,0), 2)
        cv2.putText(image, stage,
                    (125,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2)


        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(55,228,60), circle_radius=2),
                                mp_drawing.DrawingSpec(color=(255,153,204),circle_radius=2)
                                 )

        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


Squats

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# 0
# Importing necessary libraries

# In[1]:


import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose


# In[15]:

#Calculating the angle between the necessary landmarks

def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)

    if angle >180.0:
        angle = 360-angle

    return angle


# # Squats Counter

# In[33]:


#Pushups counter
cap = cv2.VideoCapture(0)

# Curl counter variables
counter = 0
stage = None

## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make detection
        results = pose.process(image)

        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark

            # Get coordinates
            hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
            knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
            ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]


            # Calculate angle
            angle = calculate_angle(hip, knee, ankle)

            # Visualize angle
            cv2.putText(image, str(angle),
                           tuple(np.multiply(hip, [640, 480]).astype(int)),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA
                                )

            # Curl counter logic
            if angle < 80:
                stage = "down"
            if angle >= 160 and stage =='down':
                stage="up"
                counter +=1
                print(counter)

        except:
            pass

        # Render curl counter
        # Setup status box
        cv2.rectangle(image, (0,0), (325,73), (245,117,16),-1)

        # Rep data
        cv2.putText(image, 'SQUATS', (15,12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,220,0), 2,cv2.LINE_8)
        cv2.putText(image, str(counter),
                    (10,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2)

        # Stage data
        cv2.putText(image, 'STAGE', (128,12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,220,0), 2)
        cv2.putText(image, stage,
                    (125,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2)


        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(55,228,60), circle_radius=2),
                                mp_drawing.DrawingSpec(color=(255,153,204),circle_radius=2)
                                 )

        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


app.py

In [ ]:
from flask import Flask,jsonify, request, render_template
from flask_cors import CORS

import subprocess

app = Flask(__name__)
CORS(app)

@app.route('/')
def welcome():
    return render_template('index.html')

@app.route('/count_exercise', methods=['POST'])
def count_exercise():
    data = request.json
    if data is None:
        return jsonify({'error': 'No JSON data received'}), 400

    exercise = data.get('exercise')
    if exercise is None:
        return jsonify({'error': 'Exercise parameter not found in JSON data'}), 400

    if exercise == 'reps':
        count = subprocess.check_output(['python', 'Reps.py']).decode().strip()
    elif exercise == 'squats':
        count = subprocess.check_output(['python', 'Squats.py']).decode().strip()
    elif exercise == 'pushups':
        count = subprocess.check_output(['python', 'Pushups.py']).decode().strip()
    else:
        count = 'Error: Invalid exercise'
    return jsonify({'count': count})

if __name__ == '__main__':
    app.run(debug=True)

index.html



<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>Exercise Counter</title>
    <style>
      body {
        font-family: Arial, sans-serif;
        margin: 0;
        padding: 0;
        background-color: #f5f5f5; /* Changed background color */
      }
      .container {
        max-width: 600px;
        margin: 20px auto;
        padding: 20px;
        background-color: #ffffff;
        border-radius: 8px;
        box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
      }
      h1 {
        text-align: center;
        margin-bottom: 20px;
        color: #333333; /* Changed text color */
      }
      form {
        text-align: center;
      }
      button {
        padding: 10px 20px;
        font-size: 16px;
        cursor: pointer;
        background-color: #007bff; /* Changed button color */
        color: #ffffff; /* Changed text color */
        border: none;
        border-radius: 4px;
      }
      button:hover {
        background-color: #0056b3; /* Darker shade on hover */
      }
      #result {
        margin-top: 20px;
        text-align: center;
        color: #555555; /* Changed text color */
      }
    </style>
  </head>
  <body>
    <div class="container">
      <h1>Exercise Counter</h1>
      <form id="exerciseForm">
        <label for="exerciseSelect">Select Exercise:</label>
        <select id="exerciseSelect" name="exercise">
          <option value="reps">Reps</option>
          <option value="squats">Squats</option>
          <option value="pushups">Pushups</option>
        </select>
        <button type="submit">Count</button>
      </form>
      <div id="result"></div>
    </div>

    <script>
      document
        .getElementById("exerciseForm")
        .addEventListener("submit", function (event) {
          event.preventDefault();
          var formData = new FormData(this);
          fetch("/count_exercise", {
            method: "POST",
            headers: {
              "Content-Type": "application/json",
              Accept: "application/json",
            },
            body: JSON.stringify(Object.fromEntries(formData)),
          })
            .then((response) => response.json())
            .then((data) => {
              document.getElementById("result").innerHTML =
                "Count: " + data.count;
            })
            .catch((error) => {
              console.error("Error:", error);
              document.getElementById("result").innerHTML =
                "Error: " + error.message;
            });
        });
    </script>
  </body>
</html>

requirements

In [ ]:
opencv-python
mediapipe
numpy
Flask
flask-cors
subprocess
cv2